### Step 1. JAVA Install

- Run First
- Kills Kernal (Intended)
- Rerun from Step 2

In [1]:
%%capture

import os
import subprocess
import textwrap

!apt-get update -y
!apt-get install -y openjdk-21-jdk-headless

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ":" + os.environ["PATH"]

In [2]:
%%capture
!pip install \
  datasets \
  transformers \
  accelerate \
  torch \
  pyserini \
  faiss-cpu \
  ragas \
  boto3 \
  litellm \
  groq -q

In [ ]:
os.kill(os.getpid(), 9)

### Step 2. Set Path, Run Imports, and Get Config
- Start From Here After Runing Step 1
- Set folder path in first cell before running
- Set values for the following keys and grant the notebook access:
  - AWS_ACCESS_KEY_ID
  - AWS_SECRET_ACCESS_KEY
  - HF_TOKEN

In [22]:
import os

# CHANGE THIS TO THE LOCATION OF THE FILE ON YOUR DRIVE
os.environ["PROJECT_ROOT"] = "/content/gdrive/MyDrive/DATASCI_210/capstone"

In [2]:
from __future__ import annotations

import json
import random
import torch
import boto3
import litellm
import asyncio
import nest_asyncio
import inspect

import numpy as np
import pandas as pd

from typing import Any, Dict, List, Optional, Tuple, Sequence, Set
from dataclasses import dataclass

from google.colab import drive, userdata

from functools import lru_cache
from collections import defaultdict
from tqdm import tqdm
from datetime import datetime
from datasets import Dataset, load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from pyserini.search.lucene import LuceneImpactSearcher, LuceneSearcher
from pyserini.encode import SpladeQueryEncoder
from pyserini.search.faiss import FaissSearcher
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision, ContextRecall, Faithfulness, AnswerAccuracy
from ragas.run_config import RunConfig

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [19]:
@dataclass(frozen=True)
class Config:
	"""Configuration.

	Attributes
	----------
	sparse_index : str
		Pyserini prebuilt index name for SPLADE sparse retrieval.
	sparse_encoder : str
		Hugging Face model id for the SPLADE query encoder.
	dense_faiss_index : str
		Pyserini prebuilt Faiss index name for dense retrieval.
	dense_encoder : str
		Hugging Face model id for dense query encoding.
	doc_searcher : str
		Pyserini prebuilt index name for document retrieval.
	reranker : str
		Hugging Face model id for cross-encoder reranking.
	bedrock_inference_profile : str
		AWS Bedrock inference profile ARN.
	aws_region : str
		AWS region for Bedrock.
	"""
	step_1_path: str
	step_1_test_path: str
	step_2_dir: str
	step_3_dir: str
	sparse_index: str
	sparse_encoder: str
	dense_faiss_index: str
	dense_encoder: str
	doc_searcher: str
	reranker: str
	aws_region: str
	critic_path: str


In [20]:
import shutil  # at top of file with other imports

def setup_config() -> Config:
    """Mount Drive and configure persistent cache directories.
    Returns
    -------
    Config
        Resolved configuration with default model/index names.
    """
    drive.mount("/content/gdrive/", force_remount=True)

    # Persistent source on Drive (survives sessions)
    pyserini_cache = os.path.join(os.environ["PROJECT_ROOT"], "pyserini_cache")
    # Fast local SSD for this session (wiped on disconnect, but fast)
    local_cache = "/content/pyserini_cache"

    os.makedirs(pyserini_cache, exist_ok=True)
    os.makedirs(local_cache, exist_ok=True)

    # Always point Pyserini to local SSD for downloading/reading
    os.environ["PYSERINI_CACHE"] = local_cache

      # If Drive has the index already, copy it to local SSD
    if os.path.exists(pyserini_cache) and os.listdir(pyserini_cache):
        if not os.path.exists(os.path.join(local_cache, "indexes")):
            print("Copying indexes from Drive to local SSD...")
            shutil.copytree(pyserini_cache, local_cache, dirs_exist_ok=True)
            print("Done copying.")
    # If local SSD has the index (just downloaded), copy back to Drive for next session
    elif os.path.exists(local_cache) and os.listdir(local_cache):
        print("Saving newly downloaded indexes to Drive for future sessions...")
        shutil.copytree(local_cache, pyserini_cache, dirs_exist_ok=True)
        print("Done saving.")

    hf_home = os.path.join(os.environ["PROJECT_ROOT"], "hf_cache")
    os.makedirs(hf_home, exist_ok=True)
    os.environ["HF_HOME"] = hf_home

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
    os.environ["GENERATION_INFERENCE_PROFILE"] = userdata.get("GENERATION_INFERENCE_PROFILE")
    os.environ["RAGAS_INFERENCE_PROFILE"] = userdata.get("RAGAS_INFERENCE_PROFILE")

    config = Config(
        step_1_path=os.path.join(os.environ["PROJECT_ROOT"], "eval", "step_1_raw_input", "hotpot_eval_300.json"),
        step_1_test_path=os.path.join(os.environ["PROJECT_ROOT"], "eval", "step_1_raw_input", "hotpot_eval_test_10.json"),
        step_2_dir=os.path.join(os.environ["PROJECT_ROOT"], "eval", "step_2_rag"),
        step_3_dir=os.path.join(os.environ["PROJECT_ROOT"], "eval", "step_3_ragas"),
        sparse_index="beir-v1.0.0-hotpotqa.splade-v3",
        sparse_encoder="naver/splade-v3",
        dense_faiss_index="beir-v1.0.0-hotpotqa.bge-base-en-v1.5",
        dense_encoder="BAAI/bge-base-en-v1.5",
        doc_searcher="beir-v1.0.0-hotpotqa.flat",
        reranker="BAAI/bge-reranker-base",
        aws_region="us-east-1",
        critic_path = os.path.join(os.environ["PROJECT_ROOT"], "src", "critic")
    )
    return config

In [5]:
hotpotqa = None

### Step 3. Load HotpotQA Data

In [6]:
def load_hotpotqa_queries() -> pd.DataFrame:
	"""Load HotpotQA (fullwiki) questions into a DataFrame.

	Returns
	-------
	pandas.DataFrame
		A DataFrame with id, split label, metadata, query text, and gold answer.
	"""
	ds = load_dataset("hotpotqa/hotpot_qa", "fullwiki")

	train_df = ds["train"].to_pandas()
	train_df["dataset"] = "train"

	val_df = ds["validation"].to_pandas()
	val_df["dataset"] = "validation"

	df = pd.concat([train_df, val_df], ignore_index=True)
	df = df.rename(columns={"question": "query", "answer": "gold_answer"})

	cols = ["id", "dataset", "type", "level", "query", "gold_answer"]
	return df[cols].copy()

### Step 4. Load Embeddings

In [7]:
@lru_cache(maxsize=1)
def get_searchers(config: Config):
	"""Initialize and cache Pyserini sparse+dense searchers.

	Parameters
	----------
	config : Config
		Resolved configuration.

	search_type : str
		Either "sparse", "dense", or "hybrid"

	Returns
	-------
	tuple
		(sparse_searcher, dense_searcher, doc_searcher)
	"""
	sparse_qe = SpladeQueryEncoder(config.sparse_encoder)
	sparse = LuceneImpactSearcher.from_prebuilt_index(config.sparse_index, sparse_qe)

	dense = FaissSearcher.from_prebuilt_index(config.dense_faiss_index, config.dense_encoder)

	doc_searcher = LuceneSearcher.from_prebuilt_index(config.doc_searcher)

	return sparse, dense, doc_searcher


### Step 5. RAG Architecture and Output

##### 5.a. Input / Output

In [8]:
def append_jsonl(jsonl_path: str, row: Dict[str, Any]) -> None:
	"""Append one JSON-serializable dict as a line to a JSONL file.

	Parameters
	----------
	jsonl_path : str
		Path to the JSONL checkpoint.
	row : Dict[str, Any]
		Row to append.

	Returns
	-------
	None
	"""
	with open(jsonl_path, "a", encoding="utf-8") as f:
		f.write(json.dumps(row, ensure_ascii=False) + "\n")
		f.flush()


def read_jsonl(jsonl_path: str) -> List[Dict[str, Any]]:
	"""Read a JSONL file into a list of dicts.

	Parameters
	----------
	jsonl_path : str
		Path to JSONL file.

	Returns
	-------
	List[Dict[str, Any]]
		Parsed JSON objects, one per line.
	"""
	rows = []
	if not os.path.exists(jsonl_path):
		return rows

	with open(jsonl_path, "r", encoding="utf-8") as f:
		for line in f:
			line = line.strip()
			if not line:
				continue
			rows.append(json.loads(line))
	return rows

##### 5.b. Retrieval

In [9]:
def get_doc_record(doc_searcher: Any, docid: str) -> Dict[str, Any]:
	"""Fetch a document record (title/text/metadata) by docid.

	Parameters
	----------
	doc_searcher : Any
		Pyserini document searcher instance.
	docid : str
		Document id.

	Returns
	-------
	dict
		Document record with keys: docid, title, text, raw_json. Fields may be None.
	"""

	out = {
		"doc_id": docid,
		"title": None,
		"text": None,
		"url": None,
	}

	try:
		doc = json.loads(doc_searcher.doc(docid).raw())
		out["doc_id"] = doc.get("_id", None)
		out["title"] = doc.get("title", None)
		out["text"] = doc.get("text", None)
		out["url"] = doc.get("metadata", {}).get("url", None)
		return out
	except Exception:
		return out

def get_retrieved_docs(
	sparse: Any,
	dense: Any,
	query: str,
	k_sparse: int = 200,
	k_dense: int = 200,
	k_fused: int = 100,
	rrf_k: int = 60,
) -> List[Tuple[str, float, Dict[str, Any]]]:
	"""Retrieve documents using hybrid retrieval and RRF fusion.

	Parameters
	----------
	sparse : Any
		LuceneImpactSearcher instance.
	dense : Any
		FaissSearcher instance.
	query : str
		Query string.
	k_sparse : int, default=200
		Top-k hits from sparse retrieval.
	k_dense : int, default=200
		Top-k hits from dense retrieval.
	k_fused : int, default=100
		Number of fused candidates to return.
	rrf_k : int, default=60
		RRF constant (larger reduces impact of top ranks).

	Returns
	-------
	list of tuple
		List of (docid, fused_score, metadata) with per-source rank/score.
	"""
	sparse_hits = sparse.search(query, k=k_sparse)
	dense_hits = dense.search(query, k=k_dense)

	fused_score: Dict[str, float] = defaultdict(float)
	meta: Dict[str, Dict[str, Any]] = {}

	def _add(hits: List[Any], key: str) -> None:
		for rank, h in enumerate(hits, start=1):
			docid = getattr(h, "docid", None)
			if docid is None:
				continue
			if docid not in meta:
				meta[docid] = {}
			fused_score[docid] += 1.0 / float(rrf_k + rank)
			meta[docid][f"{key}_rank"] = int(rank)
			meta[docid][f"{key}_score"] = float(getattr(h, "score", 0.0))

	_add(sparse_hits, "sparse")
	_add(dense_hits, "dense")

	ordered = sorted(fused_score.items(), key=lambda x: x[1], reverse=True)[:k_fused]
	return [(docid, float(score), meta.get(docid, {})) for docid, score in ordered]

##### 5.c. Reranking

In [10]:
@lru_cache(maxsize=1)
def get_reranker(reranker_name: str):
	"""Load and cache a cross-encoder reranker.

	Parameters
	----------
	reranker_name : str
		Hugging Face model id for the reranker.

	Returns
	-------
	tuple
		(tokenizer, model) ready for inference.
	"""
	tokenizer = AutoTokenizer.from_pretrained(reranker_name)
	model = AutoModelForSequenceClassification.from_pretrained(reranker_name)
	model.eval()

	device = "cuda" if torch.cuda.is_available() else "cpu"
	model.to(device)
	return tokenizer, model

@torch.no_grad()
def rerank(
	query: str,
	candidates: List[Dict[str, Any]],
	reranker_name: str,
	top_n: int = 20,
	max_length: int = 512,
	batch_size: int = 32,
) -> List[Dict[str, Any]]:
	"""Rerank candidates with a cross-encoder.

	Parameters
	----------
	query : str
		Query string.
	candidates : list of dict
		Candidate records that must include 'docid' and 'text'.
	reranker_name : str
		Hugging Face model id for the reranker.
	top_n : int, default=20
		Number of reranked candidates to return.
	max_length : int, default=512
		Max token length for the cross-encoder.
	batch_size : int, default=32
		Batch size for reranker inference.

	Returns
	-------
	list of dict
		Top reranked candidate records, each augmented with 'rerank_score'.
	"""
	tokenizer, model = get_reranker(reranker_name)

	pairs = [(query, c.get("text")) for c in candidates]
	scores: List[float] = []

	for i in range(0, len(pairs), batch_size):
		chunk = pairs[i:i + batch_size]
		enc = tokenizer(
			chunk,
			padding=True,
			truncation=True,
			max_length=max_length,
			return_tensors="pt",
		)
		enc = {k: v.to(model.device) for k, v in enc.items()}
		logits = model(**enc).logits.squeeze(-1).float().detach().cpu().numpy()
		if logits.ndim == 0:
			logits = np.array([float(logits)])
		scores.extend([float(x) for x in logits])

	out = []
	for c, s in zip(candidates, scores):
		cc = dict(c)
		cc["rerank_score"] = float(s)
		out.append(cc)

	out.sort(key=lambda x: x["rerank_score"], reverse=True)
	return out[:top_n]

##### 5.d. Prompts

**System and User Prompt**

In [54]:
from typing import Any, Dict, List

def build_resp_sys_prompt() -> str:
	"""Build the system prompt for response generation.

	Returns
	-------
	str
		System prompt string.
	"""
	return (
		"You are a careful, factual question-answering assistant.\n"
		"You must answer using ONLY the information in the provided CONTEXT.\n"
		"If the CONTEXT does not contain enough information to answer, reply "
		"exactly: I do not know.\n\n"
		"Rules:\n"
		"- Do not use outside knowledge.\n"
		"- Do not guess or infer beyond what is directly supported.\n"
		"- If the question has multiple parts, answer only the parts supported "
		"by the CONTEXT; otherwise say: I do not know.\n"
		"- If you cannot cite at least one source for the answer, reply "
		"exactly: I do not know.\n\n"
		    "- Do not include citations in answer.\n\n"
        "Answer Format:\n"
        "- Answer the question using ONLY the minimal necessary information.\n"
        "- Provide just what is asked - a single entity, name, or noun phrase.\n"
        "- Do not explain, elaborate, or add introductory phrases.\n"
        "- Do not repeat the question or use complete sentences.\n"
        "- Do not include the period mark '.' at the end of the answer.\n"
        "- Do not add any additional words beyond the direct answer.\n\n"
        "Examples:\n"
        "Q: What role did Ben Keaton play in the british film East is East written by Ayub Khan-Din\n"
        "A: a priest\n\n"
        "Q: What Mexican actress of film, television, and theatre was a protagonist in La Heredera\n"
        "A: Silvia Navarro\n\n"
        "Q: Bill Bryson grew up in a city that is the seat of which county\n"
        "A: Polk County\n\n"
        "Output:\n"
        "- Output ONLY the answer text.\n"
        "- No headings, no preamble, no explanations.\n"
    )


def build_resp_user_prompt(query: str, contexts: List[Dict[str, Any]]) -> str:
	"""Build the user prompt with a structured, delimited context block.

	Parameters
	----------
	query : str
		User question.
	contexts : list of dict
		Context passages. Each dict may contain ``title`` and ``text``.

	Returns
	-------
	str
		User prompt string containing QUESTION, CONTEXT, and an ANSWER stub.
	"""
	lines: List[str] = []
	for i, c in enumerate(contexts, start=1):
		doc_id = (c.get("doc_id") or "").strip()
		title = (c.get("title") or "").strip()
		text = (c.get("text") or "").strip()
		if title:
			lines.append(f"[{i}] {title}\n{text}")
		else:
			lines.append(f"[{i}]\n{text}")

	context_block = "\n\n".join(lines) if lines else "(no context provided)"

	return (
		"QUESTION:\n"
		f"{query}\n\n"
		"CONTEXT (numbered sources):\n"
		"---BEGIN CONTEXT---\n"
		f"{context_block}\n"
		"---END CONTEXT---\n\n"
		"ANSWER:\n"
	)


**Example question decomposition and critic prompts (NOT CURRENTLY PART OF PIPELINE)**

In [55]:
from typing import Any, Dict, List

def _format_contexts_for_critic(contexts: List[Dict[str, Any]]) -> str:
	lines: List[str] = []
	for i, c in enumerate(contexts, start=1):
		title = (c.get("title") or "").strip()
		text = (c.get("text") or "").strip()
		if title:
			#lines.append(f"[{i}] {title}\n{text}")
	 		# Change numbered sources [1], [2] to Source A, Source B:,...
			lines.append(f"Source {chr(64+i)}: {title}\n{text}")   # Source A:, Source B:, ...

			# Add reminder before ANSWER stub
			"(Output ONLY the answer entity. No citations. No period.)\n"
			"ANSWER:\n"

		else:
			lines.append(f"[{i}]\n{text}")
	return "\n\n".join(lines) if lines else "(no contexts provided)"

def build_decompose_sys_prompt() -> str:
	return (
		"You are a careful multi-hop question decomposition assistant.\n"
		"Given a QUESTION, produce a structured decomposition for retrieval.\n"
		"Return ONLY JSON.\n\n"
		"JSON schema:\n"
		"{\n"
		"  \"hops\": [\"subquestion 1\", \"subquestion 2\", ...],\n"
		"  \"query_variants\": [\"search query 1\", \"search query 2\", ...]\n"
		"}\n\n"
		"Rules:\n"
		"- 2 to 4 hops is typical.\n"
		"- query_variants should include the original QUESTION plus improved search queries.\n"
		"- Avoid overly long queries.\n"
		"- Output valid JSON only.\n"
	)


def build_decompose_user_prompt(question: str) -> str:
	return (
		"QUESTION:\n"
		f"{question}\n\n"
		"Return ONLY the JSON object."
	)

def build_critic_sys_prompt() -> str:
	return (
		"You are a Retrieval-Aware Adversarial RAG critic and planner.\n"
		"You will be given:\n"
		"- QUESTION\n"
		"- CURRENT_ANSWER\n"
		"- CONTEXTS (numbered sources)\n\n"
		"Your job:\n"
		"1) Check if CURRENT_ANSWER is fully supported by CONTEXTS and answers the QUESTION.\n"
		"2) If incomplete or unsupported, propose adversarial retrieval queries to fill gaps "
		"or resolve ambiguity.\n"
		"3) Be transparent and strict: if the answer cannot be supported from the contexts, "
		"say so.\n\n"
		"Return ONLY valid JSON with this schema:\n"
		"{\n"
		"  \"decision\": \"accept\" | \"revise\" | \"give_up\",\n"
		"  \"issues\": [\"...\"],\n"
		"  \"missing_info\": [\"...\"],\n"
		"  \"new_queries\": [\"...\"],\n"
		"  \"suggested_strategy\": \"none\" | \"increase_k\" | \"decompose\" | \"disambiguate\" "
		"| \"counter_check\",\n"
		"  \"scores\": {\n"
		"	\"groundedness\": 0.0,\n"
		"	\"completeness\": 0.0,\n"
		"	\"overall_confidence\": 0.0\n"
		"  }\n"
		"}\n\n"
		"Guidelines:\n"
		"- groundedness: are claims supported by CONTEXTS?\n"
		"- completeness: does the answer cover all parts of QUESTION?\n"
		"- overall_confidence: your holistic estimate.\n"
		"- new_queries should be short and retrieval-friendly.\n"
		"- If CONTEXTS are empty and you cannot ground an answer, decision should be \"revise\" "
		"or \"give_up\".\n"
		"- Use \"give_up\" when no reasonable new queries can help (e.g., question too "
		"underspecified).\n"
	)


def build_critic_user_prompt(question: str, answer: str, contexts: List[Dict[str, Any]]) -> str:
	return (
		"QUESTION:\n"
		f"{question}\n\n"
		"CURRENT_ANSWER:\n"
		f"{answer}\n\n"
		"CONTEXTS (numbered sources):\n"
		"---BEGIN CONTEXT---\n"
		f"{_format_contexts_for_critic(contexts)}\n"
		"---END CONTEXT---\n\n"
		"Return ONLY the JSON object."
	)


##### 5.e. Response Generation

In [13]:
@lru_cache(maxsize=1)
def get_bedrock_runtime_client(aws_region: str) -> Any:
	"""Create and cache a Bedrock Runtime boto3 client.

	Parameters
	----------
	aws_region : str
		AWS region name.

	Returns
	-------
	Any
		A boto3 Bedrock Runtime client instance.
	"""
	return boto3.client("bedrock-runtime", region_name=aws_region)

def call_llm(
	system_prompt: str,
	user_prompt: str,
	aws_region: str,
	temperature: float = 0.2,
) -> str:
	"""Call an AWS Bedrock chat model via the Converse API and return assistant text.

	Parameters
	----------
	system_prompt : str
		System instruction prompt.
	user_prompt : str
		User prompt content.
	aws_region : str
		AWS region for the Bedrock client.
	temperature : float, default=0.2
		Sampling temperature.

	Returns
	-------
	str
		Assistant text extracted from the response.
	"""
	client = get_bedrock_runtime_client(aws_region)

	messages = [
		{
			"role": "user",
			"content": [
				{"text": user_prompt},
			],
		}
	]

	system = [
		{"text": system_prompt},
	]

	request_params = {
		"modelId": os.environ["GENERATION_INFERENCE_PROFILE"],
		"system": system,
		"messages": messages,
		"inferenceConfig": {
			"temperature": float(temperature)
		},
	}

	resp = client.converse(**request_params)
	text = resp.get("output", {}).get("message", {}).get("content", [])
	return text[0].get("text", [])


##### 5.f. Critic Loop

In [43]:
import importlib.util, os, sys

PROJECT_ROOT = os.environ["PROJECT_ROOT"]

def load_package(package_name, package_path):
    spec = importlib.util.spec_from_file_location(
        package_name,
        os.path.join(package_path, "__init__.py"),
        submodule_search_locations=[package_path]
    )
    module = importlib.util.module_from_spec(spec)
    sys.modules[package_name] = module
    spec.loader.exec_module(module)
    return module

load_package("src", os.path.join(PROJECT_ROOT, "src"))
load_package("src.critic", os.path.join(PROJECT_ROOT, "src/critic"))

from src.critic.integration import integrate_critic_with_rag, process_questions_with_critic
print("✓ Imported successfully")

✓ Imported successfully


In [44]:
from src.critic.integration import integrate_critic_with_rag, process_questions_with_critic

def make_critic_retrieval_adapter(config, use_rerank=False, **pipeline_kwargs):
    """Creates a retrieval function for the critic system using your existing pipeline."""
    sparse, dense, doc_searcher = get_searchers(config)

    def retrieval_fn(query: str):
        retrieved_docs = get_retrieved_docs(
            sparse=sparse,
            dense=dense,
            query=query,
            k_sparse=pipeline_kwargs.get("k_sparse", 100),
            k_dense=pipeline_kwargs.get("k_dense", 100),
            k_fused=pipeline_kwargs.get("k_fused", 10),
            rrf_k=pipeline_kwargs.get("rrf_k", 50),
        )

        standardized = []
        for docid, fused_score, src_meta in retrieved_docs:
            doc_rec = get_doc_record(doc_searcher=doc_searcher, docid=docid)
            standardized.append({
                "text": doc_rec.get("text") or "",
                "source": doc_rec.get("url") or doc_rec.get("title") or docid,
                "score": float(fused_score)
            })
        return standardized

    return retrieval_fn


def make_critic_generation_adapter(config, temperature=0.0):
    """Creates a generation function for the critic system using your Bedrock LLM."""
    def generation_fn(question: str, context: str) -> str:
        system_prompt = build_resp_sys_prompt()  # reuse your existing prompt builder
        user_prompt = build_resp_user_prompt(    # reuse your existing prompt builder
            query=question,
            contexts=[{"text": context, "title": "", "url": ""}]
        )
        return call_llm(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            aws_region=config.aws_region,
            temperature=temperature,
        )
    return generation_fn

In [ ]:
def run_critic_on_checkpoint(
    ckpt_path: str,
    output_filename: str,
    config,
    use_rerank: bool = False,
    pipeline_kwargs: dict = None,
    n_eval: int = None,
    enable_logging: bool = False,
) -> str:
    """
    Load an existing RAG checkpoint and enhance answers with the critic loop.
    Saves results to a new JSONL file. Resumable.
    """
    pipeline_kwargs = pipeline_kwargs or {}

    # Load baseline results
    rows = read_jsonl(ckpt_path)
    if n_eval:
        rows = rows[:n_eval]

    # Output path
    output_path = os.path.join(config.step_2_dir, f"{output_filename}.jsonl")
    done_rows = read_jsonl(output_path)
    done_ids = {int(r["id"]) for r in done_rows if "id" in r}

    # Build adapters once (reuses same searcher session)
    retrieval_fn = make_critic_retrieval_adapter(config, use_rerank=use_rerank, **pipeline_kwargs)
    generation_fn = make_critic_generation_adapter(config, temperature=pipeline_kwargs.get("temperature", 0.0))

    for row in tqdm(rows, desc=f"Running critic ({output_filename})"):
        row_id = int(row["id"])
        if row_id in done_ids:
            continue

        question = row["question"]
        baseline_answer = row["answer"]

        try:
            final_answer = integrate_critic_with_rag(
                question=question,
                baseline_answer=baseline_answer,
                retriever=retrieval_fn,
                llm=generation_fn,
                enable_logging=enable_logging,
            )
        except Exception as e:
            print(f"[id={row_id}] Critic failed: {e}, falling back to baseline")
            final_answer = baseline_answer

        out_row = {
            "id": row_id,
            "question": question,
            "baseline_answer": baseline_answer,
            "answer": final_answer,           # critic-enhanced answer
            "ground_truth": row.get("ground_truth"),
            "answer_changed": baseline_answer != final_answer,
            "use_rerank": use_rerank,
        }
        append_jsonl(output_path, out_row)

    return output_path

#####5.g. Pipeline

In [23]:
def run_pipeline(
	query_df: pd.DataFrame,
	config: "Config",
	use_rerank: bool,
	k_sparse: int = 100,
	k_dense: int = 100,
	k_fused: int = 80,
	k_rerank: int = 10,
	rrf_k: int = 50,
	max_length: int = 512,
	batch_size: int = 32,
	temperature: float = 0.2,
) -> Dict[str, Any]:
	"""Run a single-query RAG pipeline (retrieve -> optional rerank -> generate).

	Parameters
	----------
	query_df : pd.DataFrame
		Single-row DataFrame with columns:
		- "id": unique query identifier
		- "query": question text
	config : Config
		Pipeline configuration (retrievers, reranker, aws_region, etc.).
	use_rerank : bool
		Whether to rerank retrieved candidates using a cross-encoder.
	k_sparse : int, default=200
		Top-k documents from sparse retrieval.
	k_dense : int, default=200
		Top-k documents from dense retrieval.
	k_fused : int, default=120
		Top-k candidates kept after fusion.
	k_rerank : int, default=30
		Top-k candidates kept after reranking (only used if use_rerank=True).
	rrf_k : int, default=60
		RRF constant (larger values reduce the impact of top ranks).
	max_length : int, default=512
		Max token length for reranker inputs.
	batch_size : int, default=32
		Batch size for reranker inference.
	temperature : float, default=0.2
		Generation temperature for the LLM call.

	Returns
	-------
	Dict[str, Any]
		Dictionary with keys:
		- "query_id": int
		- "query": str
		- "answer": str
		- "contexts": List[Dict[str, Any]] (metadata per context; includes "text")
	"""
	required_cols = {"id", "query"}
	missing = required_cols - set(query_df.columns)
	if missing:
		raise ValueError(f"query_df missing required columns: {sorted(missing)}")
	if len(query_df) != 1:
		raise ValueError("run_pipeline expects query_df to have exactly 1 row.")

	query_id = int(query_df["id"].iloc[0])
	query = str(query_df["query"].iloc[0])

	sparse, dense, doc_searcher = get_searchers(config)

	retrieved_docs = get_retrieved_docs(
		sparse=sparse,
		dense=dense,
		query=query,
		k_sparse=k_sparse,
		k_dense=k_dense,
		k_fused=k_fused,
		rrf_k=rrf_k,
	)

	contexts = []
	for docid, fused_score, src_meta in retrieved_docs:
		doc_rec = get_doc_record(doc_searcher=doc_searcher, docid=docid)
		contexts.append(
			{
				"query_id": query_id,
				"query": query,
				"doc_id": docid,
				"title": doc_rec.get("title"),
				"text": doc_rec.get("text"),
				"url": doc_rec.get("url"),
				"sparse_rank": src_meta.get("sparse_rank"),
				"sparse_score": src_meta.get("sparse_score"),
				"dense_rank": src_meta.get("dense_rank"),
				"dense_score": src_meta.get("dense_score"),
				"fused_score": float(fused_score),
			}
		)

	if use_rerank:
		contexts = rerank(
			query=query,
			candidates=contexts,
			reranker_name=config.reranker,
			top_n=k_rerank,
			max_length=max_length,
			batch_size=batch_size,
		)

	system_prompt = build_resp_sys_prompt()
	user_prompt = build_resp_user_prompt(query=query, contexts=contexts)

	response = call_llm(
		system_prompt=system_prompt,
		user_prompt=user_prompt,
		aws_region=config.aws_region,
		temperature=temperature,
	)

	return {
		"query_id": query_id,
		"query": query,
		"answer": response,
		"contexts": [
			{kk: vv for kk, vv in ctx.items() if kk not in {"query_id", "query"}}
			for ctx in contexts
		],
	}


def save_rag_pipeline_results(
	config: "Config",
	filename: str,
	use_rerank: bool,
	n_eval: Optional[int] = None,
	k_sparse: int = 100,
	k_dense: int = 100,
	k_fused: int = 60,
	k_rerank: int = 10,
	rrf_k: int = 50,
	max_length: int = 512,
	batch_size: int = 32,
	temperature: float = 0.0,
) -> str:
	"""Run the RAG pipeline over an eval dataset and checkpoint results to JSONL.

	This function is resumable: it will skip ids already present in the output
	JSONL checkpoint.

	Parameters
	----------
	config : Config
		Pipeline configuration. Must provide:
		- step_1_path: path to eval JSON (list of dicts with "question", "answer")
		- step_2_dir: directory where JSONL checkpoints should be written
		- aws_region, reranker, etc. for downstream calls
	filename : str
		Output checkpoint base name (written to "<step_2_dir>/<filename>.jsonl").
	use_rerank : bool
		Whether to rerank retrieved candidates using a cross-encoder.
	n_eval : Optional[int], default=None
		Number of examples to evaluate. If None, evaluates the entire dataset.
	k_sparse : int, default=100
		Top-k documents from sparse retrieval.
	k_dense : int, default=100
		Top-k documents from dense retrieval.
	k_fused : int, default=60
		Top-k candidates kept after fusion.
	k_rerank : int, default=10
		Top-k candidates kept after reranking (only used if use_rerank=True).
	rrf_k : int, default=50
		RRF constant.
	max_length : int, default=512
		Max token length for reranker inputs.
	batch_size : int, default=32
		Batch size for reranker inference.
	temperature : float, default=0.0
		Generation temperature for the LLM call.

	Returns
	-------
	str
		Path to the JSONL checkpoint file that was written.
	"""
	#with open(config.step_1_path, "r", encoding="utf-8") as f:
	with open(config.step_1_test_path, "r", encoding="utf-8") as f:
		eval_ds_full = json.load(f)

	if not isinstance(eval_ds_full, list):
		raise ValueError("Expected step_1_path JSON to be a list of examples.")

	if n_eval is None or n_eval >= len(eval_ds_full):
		eval_ds = eval_ds_full
	else:
		eval_ds = eval_ds_full[:n_eval]

	os.makedirs(config.step_2_dir, exist_ok=True)
	ckpt_path = os.path.join(config.step_2_dir, f"{filename}.jsonl")

	rows = read_jsonl(ckpt_path)
	done_ids = {int(r["id"]) for r in rows if "id" in r}

	pipeline_kwargs = {
		"config": config,
		"use_rerank": use_rerank,
		"k_sparse": k_sparse,
		"k_dense": k_dense,
		"k_fused": k_fused,
		"k_rerank": k_rerank,
		"rrf_k": rrf_k,
		"max_length": max_length,
		"batch_size": batch_size,
		"temperature": temperature,
	}

	total = len(eval_ds)
	for idx, sample in tqdm(
		enumerate(eval_ds),
		total=total,
		desc=f"Running RAG pipeline ({filename})",
	):
		if idx in done_ids:
			continue

		query_df = pd.DataFrame([{"id": idx, "query": sample["question"]}])

		resp = run_pipeline(query_df=query_df, **pipeline_kwargs)

		row = {
			"id": idx,
			"question": resp.get("query", sample["question"]),
			"answer": resp["answer"],
			"contexts": [c.get("text") for c in resp["contexts"]],
			"ground_truth": sample.get("answer"),
			"use_rerank": use_rerank,
		}
		append_jsonl(ckpt_path, row)

	return ckpt_path


##### 5.h. Ragas

In [ ]:
def evaluate_rag_pipeline(
	input_path: str,
	output_path: str,
	run_config: "RunConfig",
	ragas_llm: Any,
	metrics: Sequence[Any],
	sleep_per_call: float = 0.5,
	base_sleep: float = 2.0,
	max_sleep: float = 60.0,
) -> pd.DataFrame:
	"""
	Evaluate RAG pipeline outputs using RAGAS metric scorers and
	incrementally persist results to disk.

	This function reads checkpointed RAG outputs (JSONL), evaluates each
	row sequentially using the provided RAGAS metrics and LLM judge,
	applies exponential backoff on throttling errors, and writes results
	incrementally to a CSV file to support resumability.

	Parameters
	----------
	input_path : str
		Path to JSONL checkpoint file containing rows with keys:
		"id", "question", "answer", "contexts", and "ground_truth".
	output_path : str
		Path to CSV file where results will be written. If the file
		already exists, previously completed rows are skipped.
	run_config : RunConfig
		Configuration object controlling retry behavior. The attribute
		`max_retries` is used for exponential backoff attempts.
	ragas_llm : Any
		RAGAS-compatible LLM instance created via `llm_factory`.
	metrics : Sequence[Any]
		Sequence of RAGAS metric classes (e.g.,
		[ContextPrecision, Faithfulness]). Each metric must expose
		an asynchronous `ascore()` method.
	sleep_per_call : float, optional
		Fixed delay (in seconds) inserted after each metric call
		to reduce Bedrock throttling. Default is 0.5.
	base_sleep : float, optional
		Initial sleep duration (seconds) used in exponential backoff.
		Default is 2.0.
	max_sleep : float, optional
		Maximum sleep duration (seconds) for exponential backoff.
		Default is 60.0.

	Returns
	-------
	pd.DataFrame
		DataFrame containing raw RAG outputs merged with computed
		metric scores. The returned DataFrame is read from the
		persisted CSV at `output_path`.
	"""
	done_ids = set()
	if os.path.exists(output_path):
		try:
			existing = pd.read_csv(output_path)
			if "id" in existing.columns:
				done_ids = set(existing["id"].dropna().astype(int).tolist())
		except Exception:
			done_ids = set()

	raw_rows = read_jsonl(input_path)
	rows: List[Dict[str, Any]] = []
	for r in raw_rows:
		rows.append(
			{
				"id": int(r["id"]),
				"question": r["question"],
				"answer": r["answer"],
				"contexts": list(r["contexts"]),
				"ground_truth": r["ground_truth"],
			}
		)

	max_tries = int(getattr(run_config, "max_retries"))

	def _is_throttle_error(exc: BaseException) -> bool:
		msg = str(exc).lower()
		return (
			"is throttling" in msg
			or "too many requests" in msg
			or "429" in msg
			or isinstance(exc, litellm.RateLimitError)
		)

	async def _ascore_with_backoff(scorer: Any, kwargs: Dict[str, Any]) -> Any:
		for attempt in range(max_tries):
			try:
				res = await scorer.ascore(**kwargs)
				if sleep_per_call > 0:
					await asyncio.sleep(sleep_per_call)
				return res
			except BaseException as e:
				if not _is_throttle_error(e) or attempt == max_tries - 1:
					raise
				# Exponential backoff + jitter
				sleep_s = min(
					max_sleep,
					(base_sleep * (2 ** attempt)) + random.uniform(0.0, 1.0),
				)
				await asyncio.sleep(sleep_s)

	async def _score_one(row: Dict[str, Any]) -> Dict[str, Any]:
		score_row: Dict[str, Any] = {
			"id": row["id"],
			"question": row["question"],
			"pred_answer": row["answer"],
			"gold_answer": row["ground_truth"],
			"num_contexts": len(row["contexts"]),
			"contexts_json": json.dumps(row["contexts"], ensure_ascii=False),
			"contexts_joined": "\n\n---\n\n".join(row["contexts"]),
		}

		base_kwargs = {
			"user_input": row["question"],
			"reference": row["ground_truth"],
			"retrieved_contexts": row["contexts"],
			"answer": row["answer"],
			"response": row["answer"],
			"ground_truth": row["ground_truth"],
			"question": row["question"],
			"contexts": row["contexts"],
		}

		for m in metrics:
			scorer = m(llm=ragas_llm)
			allowed = set(inspect.signature(scorer.ascore).parameters.keys())
			kwargs = {k: v for k, v in base_kwargs.items() if k in allowed}

			res = await _ascore_with_backoff(scorer, kwargs)

			name = getattr(scorer, "name", scorer.__class__.__name__)
			score_row[name] = getattr(res, "value", res)

		return score_row

	def _run_coroutine(coro: Any) -> Any:
		try:
			loop = asyncio.get_running_loop()
		except RuntimeError:
			loop = None

		if loop is None:
			return asyncio.run(coro)

		nest_asyncio.apply()
		return loop.run_until_complete(coro)

	out_dir = os.path.dirname(output_path)
	if out_dir:
		os.makedirs(out_dir, exist_ok=True)

	write_header = not os.path.exists(output_path)
	for row in tqdm(rows, desc="RAGAS scoring", total=len(rows)):
		if row["id"] in done_ids:
			continue

		scored = _run_coroutine(_score_one(row))
		pd.DataFrame([scored]).to_csv(
			output_path,
			mode="a",
			header=write_header,
			index=False,
		)
		write_header = False
		done_ids.add(row["id"])

	return pd.read_csv(output_path)

### Step 6. Run Pipeline

##### 6.a. Load data and config

In [48]:
%%capture
if hotpotqa is None:
    config = setup_config()
    hotpotqa = load_hotpotqa_queries()

##### 6.b. Run RAG Pipeline

In [49]:
n_eval = None

retrieval_only_filename = "retrieval_only"
retrieve_rerank_filename = "retrieve_rerank"

pipeline_kwargs_retrieval_only = {
    "k_sparse": 100,
    "k_dense": 100,
    "k_fused": 10,
    "k_rerank": 10,
    "rrf_k": 50,
    "max_length": 512,
    "batch_size": 32,
    "temperature": 0.0
}

pipeline_kwargs_retrieve_rerank = {
    "k_sparse": 100,
    "k_dense": 100,
    "k_fused": 80,
    "k_rerank": 10,
    "rrf_k": 50,
    "max_length": 512,
    "batch_size": 32,
    "temperature": 0.0
}

In [ ]:
#!rm -rf "/content/gdrive/MyDrive/DATASCI_210/capstone/pyserini_cache/indexes/faiss-flat.beir-v1.0.0-hotpotqa.bge-base-en-v1.5.20240107.d2c08665e8cd750bd06ceb7d23897c94"
#!rm -rf "/content/pyserini_cache/indexes/faiss-flat.beir-v1.0.0-hotpotqa.bge-base-en-v1.5.20240107.d2c08665e8cd750bd06ceb7d23897c94"

In [59]:
retrieval_only_ckpt = save_rag_pipeline_results(
	config=config,
  n_eval=n_eval,
	filename=retrieval_only_filename,
	use_rerank=False,
  **pipeline_kwargs_retrieval_only
)

retrieve_rerank_ckpt = save_rag_pipeline_results(
	config=config,
  n_eval=n_eval,
	filename=retrieve_rerank_filename,
	use_rerank=True,
  **pipeline_kwargs_retrieve_rerank
)


Running RAG pipeline (retrieve_rerank):   0%|          | 0/10 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: naver/splade-v3
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Running RAG pipeline (retrieve_rerank): 100%|██████████| 10/10 [13:01<00:00, 78.15s/it]


In [57]:
data = []

file_path = "/content/gdrive/MyDrive/DATASCI_210/capstone/eval/step_2_rag/retrieve_rerank.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))




In [58]:
import pandas as pd

# Convert critic checkpoints to CSV
retrieval_only_df = pd.DataFrame(read_jsonl(retrieval_only_ckpt))
retrieve_rerank_df = pd.DataFrame(read_jsonl(retrieve_rerank_ckpt))

# Save to CSV
retrieval_only_df.to_csv(
    os.path.join(config.step_2_dir, "retrieval_only.csv"), index=False
)
retrieve_rerank_df.to_csv(
    os.path.join(config.step_2_dir, "retrieve_rerank.csv"), index=False
)

print(f"Rows: {len(retrieve_rerank_df)}")
print(retrieve_rerank_df[['contexts', 'id', 'question', 'ground_truth', 'use_rerank', 'answer']].head(3))

Rows: 10
                                            contexts  id  \
0  [Macbeth (] ) is an opera in four acts by Gius...   0   
1  [Gregori Aleksandrovich Margulis (Russian: Гри...   1   
2  [Eric Banadinović (born 9 August 1968), known ...   2   

                                            question  \
0  Repertoire of Plácido Domingo appeared in an o...   
1  Who was born first, Grigory Margulis or Leonid...   
2  Which Star of British-Canadian-American satiri...   

                      ground_truth  use_rerank  \
0                   Giuseppe Verdi        True   
1  Gregori Aleksandrovich Margulis        True   
2                 Eric Banadinović        True   

                                              answer  
0  Macbeth, an opera written by Giuseppe Verdi [1...  
1  Grigory Margulis was born first, on February 2...  
2                                         Eric Bana.  


In [ ]:
columns = list({key for item in data for key in item.keys()})
print(columns)

['contexts', 'id', 'question', 'ground_truth', 'use_rerank', 'answer']


##### 6.c. Run Critic Loop

In [ ]:
# Enhance retrieval-only results with critic
critic_retrieval_only_ckpt = run_critic_on_checkpoint(
    ckpt_path=retrieval_only_ckpt,
    output_filename="critic_retrieval_only",
    config=config,
    use_rerank=False,
    pipeline_kwargs=pipeline_kwargs_retrieval_only,
    n_eval=n_eval,
    enable_logging=False,   # set True to debug a single question
)

# Enhance retrieve+rerank results with critic
critic_retrieve_rerank_ckpt = run_critic_on_checkpoint(
    ckpt_path=retrieve_rerank_ckpt,
    output_filename="critic_retrieve_rerank",
    config=config,
    use_rerank=True,
    pipeline_kwargs=pipeline_kwargs_retrieve_rerank,
    n_eval=n_eval,
    enable_logging=False,
)

print(f"Critic retrieval-only results: {critic_retrieval_only_ckpt}")
print(f"Critic retrieve+rerank results: {critic_retrieve_rerank_ckpt}")

config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: naver/splade-v3
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

lucene-inverted.beir-v1.0.0-hotpotqa.flat.20221116.505594.tar.gz: 100%|██████████| 1.88G/1.88G [00:19<00:00, 101MB/s]
Running critic (critic_retrieval_only):  10%|█         | 1/10 [00:00<00:04,  1.81it/s]ERROR:src.critic.decomposer:Unexpected error during decomposition: Missing required environment variable: GROQ_MODEL
ERROR:src.critic.critic_system:Unexpected error in critic loop: Missing required environment variable: GROQ_MODEL
ERROR:src.critic.decomposer:Unexpected error during decomposition: Missing required environment variable: GROQ_MODEL
ERROR:src.critic.critic_system:Unexpected error in critic loop: Missing required environment variable: GROQ_MODEL
ERROR:src.critic.decomposer:Unexpected error during decomposition: Missing required environment variable: GROQ_MODEL
ERROR:src.critic.critic_system:Unexpected error in critic loop: Missing required environment variable: GROQ_MODEL
ERROR:src.critic.decomposer:Unexpected error during decomposition: Missing required environment variabl

Critic retrieval-only results: /content/gdrive/MyDrive/DATASCI_210/capstone/eval/step_2_rag/critic_retrieval_only.jsonl
Critic retrieve+rerank results: /content/gdrive/MyDrive/DATASCI_210/capstone/eval/step_2_rag/critic_retrieve_rerank.jsonl


In [ ]:
data_critic = []

file_path = "/content/gdrive/MyDrive/DATASCI_210/capstone/eval/step_2_rag/critic_retrieve_rerank.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data_critic.append(json.loads(line))

In [ ]:
columns = list({key for item in data_critic for key in item.keys()})
print(columns)

['baseline_answer', 'use_rerank', 'id', 'question', 'answer', 'answer_changed', 'ground_truth']


In [ ]:
import pandas as pd

# Convert critic checkpoints to CSV
critic_retrieval_only_df = pd.DataFrame(read_jsonl(critic_retrieval_only_ckpt))
critic_retrieve_rerank_df = pd.DataFrame(read_jsonl(critic_retrieve_rerank_ckpt))

# Save to CSV
critic_retrieval_only_df.to_csv(
    os.path.join(config.step_2_dir, "critic_retrieval_only.csv"), index=False
)
critic_retrieve_rerank_df.to_csv(
    os.path.join(config.step_2_dir, "critic_retrieve_rerank.csv"), index=False
)

print(f"Rows: {len(critic_retrieve_rerank_df)}")
print(critic_retrieve_rerank_df[["id","question","baseline_answer","answer","answer_changed"]].head(3))

Rows: 10
   id                                           question  \
0   0  Repertoire of Plácido Domingo appeared in an o...   
1   1  Who was born first, Grigory Margulis or Leonid...   
2   2  Which Star of British-Canadian-American satiri...   

                                     baseline_answer  \
0  Macbeth, an opera written by Giuseppe Verdi [1...   
1  Grigory Margulis was born first, on February 2...   
2                                         Eric Bana.   

                                              answer  answer_changed  
0  Macbeth, an opera written by Giuseppe Verdi [1...           False  
1  Grigory Margulis was born first, on February 2...           False  
2                                         Eric Bana.           False  


In [ ]:
critic_retrieve_rerank_ckpt

'/content/gdrive/MyDrive/DATASCI_210/capstone/eval/step_2_rag/critic_retrieve_rerank.jsonl'

##### 6.c. RUN Ragas Evaluation

In [ ]:
#with open(config.step_1_path, "r", encoding="utf-8") as f:
with open(config.step_1_test_path, "r", encoding="utf-8") as f:
		eval_df_full = pd.DataFrame(json.load(f))[["id", "question", "type", "level"]]

In [ ]:
%%capture
temperature=0.0
max_tokens = 2048

ragas_run_config = RunConfig(
	max_workers=1,
	timeout=540,
	max_retries=3,
)

ragas_llm = llm_factory(
	f"bedrock/{os.environ['RAGAS_INFERENCE_PROFILE']}",
	provider="litellm",
	client=litellm.acompletion,
	temperature=temperature,
  max_tokens=max_tokens,
  system_prompt=(
		"Output ONLY valid JSON for the required schema. No explanations."
	),
	response_format={"type": "json_object"},
)

metrics = [ContextPrecision, ContextRecall, Faithfulness, AnswerAccuracy]

In [ ]:
retrieval_only_results = evaluate_rag_pipeline(
    input_path=retrieval_only_ckpt,
    output_path=os.path.join(config.step_3_dir, f"{retrieval_only_filename}.csv"),
    run_config=ragas_run_config,
    ragas_llm=ragas_llm,
    metrics=metrics,
)

RAGAS scoring: 100%|██████████| 10/10 [00:00<00:00, 191520.73it/s]


In [ ]:
retrieval_only_results.head(10)

,id,question,pred_answer,gold_answer,num_contexts,contexts_json,contexts_joined,context_precision,context_recall,faithfulness,answer_accuracy
0,0,Repertoire of Plácido Domingo appeared in an o...,"Giuseppe Verdi [3], [5], [9]",Giuseppe Verdi,10,"[""Spanish tenor Plácido Domingo has officially...",Spanish tenor Plácido Domingo has officially s...,0.871111,1.0,1.000000,1.00
1,1,"Who was born first, Grigory Margulis or Leonid...","Grigory Margulis was born first, on February 2...",Gregori Aleksandrovich Margulis,10,"[""Leonid Anatolievich Levin ( ; Russian: Леони...",Leonid Anatolievich Levin ( ; Russian: Леони́д...,0.500000,1.0,1.000000,0.25
2,2,Which Star of British-Canadian-American satiri...,"Eric Bana, a star of the British-Canadian-Amer...",Eric Banadinović,10,"[""Full Frontal was an Australian sketch comedy...",Full Frontal was an Australian sketch comedy s...,0.666667,1.0,1.000000,0.25
3,3,"What ""sport"" involving animals dates all the w...","Cricket fighting, a blood sport involving the ...",Cricket fighting,10,"[""Keeping crickets as pets emerged in China in...",Keeping crickets as pets emerged in China in e...,0.684524,1.0,0.666667,0.75
4,4,"Which Houston, TX based store did Paul Traub p...",I do not know.,Stage Stores,10,"[""Paul R. Traub (born January 31, 1952, Brookl...","Paul R. Traub (born January 31, 1952, Brooklyn...",0.633333,1.0,0.500000,0.00
5,5,Kevin Watson currently manages which Southern ...,Kevin Watson currently manages Bishop's Stortf...,Bishop's Stortford,10,"[""Kevin Edward Watson (born 3 January 1974) is...",Kevin Edward Watson (born 3 January 1974) is a...,1.000000,1.0,0.500000,0.75
6,6,What American filmmaker was the co-founder of ...,"Stanley Lloyd Kaufman, Jr.","Stanley Lloyd Kaufman, Jr.",10,"[""Stanley Lloyd Kaufman, Jr. (born December 30...","Stanley Lloyd Kaufman, Jr. (born December 30, ...",1.000000,1.0,1.000000,1.00
7,7,The Legendary A&M Sessions contain five songs ...,Captain Beefheart & His Magic Band [1],Magic Band,10,"[""The Legendary A&M Sessions is an extended pl...",The Legendary A&M Sessions is an extended play...,1.000000,1.0,1.000000,0.75
8,8,"Of all the teams John Nyskohus played for, whi...",I do not know.,Adelaide City,10,"[""John Nyskohus (born 15 October 1951) is an A...",John Nyskohus (born 15 October 1951) is an Aus...,1.000000,1.0,0.500000,0.00
9,9,What is the translation of the name of the riv...,I do not know.,boundary river,10,"[""Widnes is an industrial town in the unitary ...",Widnes is an industrial town in the unitary au...,0.950000,1.0,0.000000,0.00


In [ ]:
retrieval_only_results.describe()

,id,num_contexts,context_precision,context_recall,faithfulness,answer_accuracy
count,10.00000,10.0,10.000000,10.0,10.000000,10.000000
mean,4.50000,10.0,0.830563,1.0,0.716667,0.475000
std,3.02765,0.0,0.190644,0.0,0.342918,0.415832
min,0.00000,10.0,0.500000,1.0,0.000000,0.000000
25%,2.25000,10.0,0.671131,1.0,0.500000,0.062500
50%,4.50000,10.0,0.910556,1.0,0.833333,0.500000
75%,6.75000,10.0,1.000000,1.0,1.000000,0.750000
max,9.00000,10.0,1.000000,1.0,1.000000,1.000000


In [ ]:
retrieve_rerank_results = evaluate_rag_pipeline(
    input_path=retrieve_rerank_ckpt,
    output_path=os.path.join(config.step_3_dir, f"{retrieve_rerank_filename}.csv"),
    run_config=ragas_run_config,
    ragas_llm=ragas_llm,
    metrics=metrics,
)

RAGAS scoring: 100%|██████████| 10/10 [00:00<00:00, 115545.56it/s]


In [ ]:
retrieve_rerank_results.head(10)

,id,question,pred_answer,gold_answer,num_contexts,contexts_json,contexts_joined,context_precision,context_recall,faithfulness,answer_accuracy
0,0,Repertoire of Plácido Domingo appeared in an o...,"Macbeth, an opera written by Giuseppe Verdi [1...",Giuseppe Verdi,10,"[""Macbeth (] ) is an opera in four acts by Giu...",Macbeth (] ) is an opera in four acts by Giuse...,1.000000,1.0,1.0,0.50
1,1,"Who was born first, Grigory Margulis or Leonid...","Grigory Margulis was born first, on February 2...",Gregori Aleksandrovich Margulis,10,"[""Gregori Aleksandrovich Margulis (Russian: Гр...",Gregori Aleksandrovich Margulis (Russian: Григ...,1.000000,1.0,1.0,0.25
2,2,Which Star of British-Canadian-American satiri...,Eric Bana.,Eric Banadinović,10,"[""Eric Banadinović (born 9 August 1968), known...","Eric Banadinović (born 9 August 1968), known p...",0.833333,1.0,1.0,0.25
3,3,"What ""sport"" involving animals dates all the w...",Cricket fighting.,Cricket fighting,10,"[""Cricket fighting is a blood sport involving ...",Cricket fighting is a blood sport involving th...,0.897222,1.0,1.0,1.00
4,4,"Which Houston, TX based store did Paul Traub p...",Sakowitz.,Stage Stores,10,"[""Paul R. Traub (born January 31, 1952, Brookl...","Paul R. Traub (born January 31, 1952, Brooklyn...",0.543889,1.0,0.5,0.00
5,5,Kevin Watson currently manages which Southern ...,Bishop's Stortford [1],Bishop's Stortford,10,"[""Kevin Edward Watson (born 3 January 1974) is...",Kevin Edward Watson (born 3 January 1974) is a...,1.000000,1.0,1.0,1.00
6,6,What American filmmaker was the co-founder of ...,"Stanley Lloyd Kaufman, Jr.","Stanley Lloyd Kaufman, Jr.",10,"[""Stanley Lloyd Kaufman, Jr. (born December 30...","Stanley Lloyd Kaufman, Jr. (born December 30, ...",0.751389,1.0,1.0,1.00
7,7,The Legendary A&M Sessions contain five songs ...,The Legendary A&M Sessions contain five songs ...,Magic Band,10,"[""The Legendary A&M Sessions is an extended pl...",The Legendary A&M Sessions is an extended play...,1.000000,1.0,1.0,0.75
8,8,"Of all the teams John Nyskohus played for, whi...",Adelaide City. [2],Adelaide City,10,"[""John Nyskohus (born 15 October 1951) is an A...",John Nyskohus (born 15 October 1951) is an Aus...,1.000000,1.0,1.0,1.00
9,9,What is the translation of the name of the riv...,"The River Mersey's name translates as ""boundar...",boundary river,10,"[""Widnes is an industrial town in the unitary ...",Widnes is an industrial town in the unitary au...,0.824008,1.0,1.0,1.00


In [ ]:
retrieve_rerank_results.describe()

,id,num_contexts,context_precision,context_recall,faithfulness,answer_accuracy
count,10.00000,10.0,10.000000,10.0,10.000000,10.000000
mean,4.50000,10.0,0.884984,1.0,0.950000,0.675000
std,3.02765,0.0,0.151611,0.0,0.158114,0.391755
min,0.00000,10.0,0.543889,1.0,0.500000,0.000000
25%,2.25000,10.0,0.826339,1.0,1.000000,0.312500
50%,4.50000,10.0,0.948611,1.0,1.000000,0.875000
75%,6.75000,10.0,1.000000,1.0,1.000000,1.000000
max,9.00000,10.0,1.000000,1.0,1.000000,1.000000


### 7. Save Final Output

In [ ]:
retrieval_only_results = pd.merge(
    eval_df_full,
    retrieval_only_results.drop(columns=["id"]),
    on="question",
    how="left"
)
retrieval_only_results.to_csv(os.path.join(config.step_3_dir, f"{retrieval_only_filename}_final.csv"), index=False)

In [ ]:
retrieve_rerank_results = pd.merge(
    eval_df_full,
    retrieve_rerank_results.drop(columns=["id"]),
    on="question",
    how="left"
)
retrieve_rerank_results.to_csv(os.path.join(config.step_3_dir, f"{retrieve_rerank_filename}_final.csv"), index=False)